In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 2.5 Scattering and the Rutherford Cross-Section

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume II — Analytical Mechanics",
    number="2.5",
    title="Scattering and the Rutherford Cross-Section",
    blurb="Unbound orbits and how a beam scatters: impact parameter, "
    "deflection angle, and the differential cross-section — recovering "
    "Rutherford's famous 1/sin⁴(θ/2) law from classical mechanics.",
    difficulty="intermediate",
    estimate="75–100 min",
)

## Notebook overview

[§2.4](central-force.ipynb) read the effective potential as a 1-D energy diagram and classified
orbits by energy: $E<0$ gave the bound ellipses, and $E\ge0$ was set aside as the
**scattering states**. This notebook is that $E>0$ sequel. A particle now comes in
from far away, swings once past the force centre on an unbound hyperbola, and
leaves: deflected through a **scattering angle** $\theta$ that depends on how
closely it was aimed, the **impact parameter** $b$.

The payoff is one of the most consequential formulas in physics. A *beam* of
particles carries a spread of impact parameters; geometry maps that spread of $b$
onto a spread of outgoing angles $\theta$, and the **differential cross-section**
$d\sigma/d\Omega$ records how many particles emerge per unit solid angle. For the
Coulomb potential this is the **Rutherford formula** $d\sigma/d\Omega\propto
1/\sin^4(\theta/2)$: the long tail of large-angle events that, measured in Geiger and
Marsden's gold-foil experiment, revealed the atomic nucleus. We derive it from the
same central-force machinery as [§2.4](central-force.ipynb), with no quantum mechanics anywhere.

We reuse the 2-D integrator from [§1.4](../01-elementary-mechanics/kepler-orbits.ipynb)/[§2.4](central-force.ipynb), add a `deflection(b, E)` that fires one
particle from far away and reports how much it turned, and put it to work: a single
hyperbolic trajectory, the deflection law $\tan(\theta/2)=k/(2Eb)$, the Rutherford
cross-section as the centrepiece, an animated **beam** fanning out by angle, an
attractive-vs-repulsive comparison, and the divergent total cross-section that
announces the infinite range of the Coulomb force.

> **A deliberate numerical lesson.** Rutherford's formulas assume the particle
> starts at *infinity*. We must launch from a finite distance, where the particle
> already feels the $1/r$ tail, so trajectory-measured angles come out
> systematically a little low, and the more so for large $b$ (those orbits skim the
> tail for longer). The trajectory checks below therefore run at $\sim2\%$
> tolerance, while the purely *analytic* cross-section check (Exercise 4) is exact
> to $\sim10^{-8}$ because it never integrates a trajectory at all. Watching where
> the error does and does not appear is half the point of the notebook.

> **How to read the checks.** Each exercise ends with a validation that compares
> a computed result to an expected physical fact. A ✗ does *not* by itself mean
> the physics is wrong: it means the output didn't match what the check
> expected, which may be a real error, a different-but-valid convention (a sign,
> a unit, an array order), or simply too tight a tolerance. Treat a ✗ as a prompt
> to locate the discrepancy; passing is strong evidence of correctness, not proof.

> **Scope.** This is a working review, not a textbook chapter. For the full
> theory of central-force scattering and the cross-section, see Nolting,
> *Theoretical Physics 2* {cite}`nolting2`, Goldstein, Poole & Safko
> {cite}`goldstein` (ch. 3), and Landau & Lifshitz, *Mechanics*
> {cite}`landau_mechanics` (§18–19).

## Theory in brief

### The scattering setup

A beam of particles, each of energy $E>0$, comes in from infinity travelling
parallel to a chosen axis. A given particle is offset from the axis through the
centre by the **impact parameter** $b$: the perpendicular distance at which it
*would* pass the centre if there were no force. The central force bends its path
into an unbound hyperbola, and it departs to infinity having turned through the
**scattering angle** $\theta$ (see {numref}`fig-scatter-geometry`). The whole
problem is to find the map $b\mapsto\theta$, and then what it does to a beam.

Because the orbit is unbound, the conserved angular momentum is fixed by the
incoming conditions: $L=\mu v_\infty b$ with $v_\infty=\sqrt{2E/\mu}$.

### The deflection angle

Integrating the radial motion {eq}`eq-veff` from the distance of closest approach
$r_{\min}$ (the single turning point) out to infinity, and doubling by symmetry,
gives the **deflection-angle integral**

```{math}
:label: eq-theta-integral
\theta(b) = \pi - 2\!\int_{r_{\min}}^{\infty}
\frac{L/r^2}{\sqrt{\,2\mu\bigl(E-V_{\mathrm{eff}}(r)\bigr)}}\;dr ,
\qquad V_{\mathrm{eff}}(r)=V(r)+\frac{L^2}{2\mu r^2}.
```

For the repulsive Coulomb potential $V(r)=k/r$ (with $k>0$) this integral has a
closed form (Landau & Lifshitz, *Mechanics* {cite}`landau_mechanics`, §19,
carries the integration out): the orbit is a hyperbola whose asymptotes make the simple relation

```{math}
:label: eq-rutherford-angle
\tan\!\frac{\theta}{2} = \frac{k}{2Eb} ,
```

so $b\to0$ (a head-on aim) gives $\theta\to\pi$ (back-scatter), and $b\to\infty$
gives $\theta\to0$ (no deflection). Inverting, $b(\theta)=\dfrac{k}{2E}\cot\dfrac{\theta}{2}$.

### The differential cross-section

Particles entering through the annulus between $b$ and $b+db$ (area $2\pi b\,|db|$)
all emerge between $\theta$ and $\theta+d\theta$ (solid angle $d\Omega=2\pi\sin\theta\,|d\theta|$).
Equating the counts defines the **differential cross-section**

```{math}
:label: eq-dsigma
\frac{d\sigma}{d\Omega} = \left|\frac{b}{\sin\theta}\,\frac{db}{d\theta}\right| .
```

Substituting $b(\theta)$ from {eq}`eq-rutherford-angle` and its derivative
$db/d\theta=-(k/4E)\csc^2(\theta/2)$ collapses everything to the **Rutherford
formula**

```{math}
:label: eq-rutherford
\frac{d\sigma}{d\Omega} = \left(\frac{k}{4E}\right)^{2}\frac{1}{\sin^{4}(\theta/2)} .
```

### What the cross-section means, and its catastrophe

$d\sigma/d\Omega$ has units of area per steradian: it converts the incident beam
flux into the rate of particles scattered into a detector at angle $\theta$. The
$1/\sin^4(\theta/2)$ divergence as $\theta\to0$ is the historic signature: a sharp
forward pile-up with a long tail of large-angle events that only a *small, hard,
charged* core could produce. Integrating it over all angles, the **total**
cross-section diverges, because the Coulomb force has infinite range: every
particle, however large its $b$, is deflected a little. A physical target screens
the charge at large $r$ and restores a finite total: the link to the screened
(Yukawa) potential and, ultimately, the quantum Born approximation of Volume VI.

---
## Setup

We work in **reduced units** $\mu=1$ and a repulsive Coulomb strength $k=1$, so
$V(r)=k/r$ and the deflection law {eq}`eq-rutherford-angle` is just
$\tan(\theta/2)=1/(2Eb)$. The reusable core is small:

- `cartesian_rhs(t, s, k)`: the 2-D Newtonian integrator, with the Coulomb
  acceleration $\mathbf a = k\,\mathbf r/r^3$ (outward for $k>0$, inward for $k<0$).
- `deflection(b, E, k)`: fires one particle from far out on the $-x$ axis at
  impact parameter $b$, integrates until it returns to its starting radius, and
  returns the outgoing velocity angle $\theta$ together with the trajectory.

A crucial detail lives in `deflection`: we start at $x_0=-\max(200,\,100\,|b|)$
(**far**, and farther still for large $b$) precisely to keep the finite-distance
systematic (flagged above) down at the percent level. The scattering geometry it
realises is sketched in {numref}`fig-scatter-geometry`.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

from ecp import draw, validate
from ecp.animate import show

MU = 1.0  # reduced units: μ = 1
K = 1.0  # repulsive Coulomb strength k (V = k/r)


def V(r, k=K):
    """Coulomb potential energy V(r) = k/r (repulsive for k > 0).

    Parameters
    ----------
    r : float or numpy.ndarray
        Separation.
    k : float, optional
        Coupling strength.

    Returns
    -------
    float or numpy.ndarray
        The potential energy.
    """
    return k / r


def cartesian_rhs(_t, s, k):
    """2-D Coulomb state derivative; acceleration a = k·r/r^3 (μ = 1)."""
    x, y, vx, vy = s
    r3 = (x * x + y * y) ** 1.5
    return [vx, vy, k * x / r3, k * y / r3]


def deflection(b, E, k=K, x0=None, n_eval=2000):
    """Integrate one scattering trajectory and return its deflection angle.

    Launches a particle at impact parameter b from far away and measures the
    asymptotic turn.

    Parameters
    ----------
    b : float
        Impact parameter.
    E : float
        Incident energy.
    k : float, optional
        Coupling.
    x0 : float, optional
        Launch distance.
    n_eval : int, optional
        Output samples.

    Returns
    -------
    float
        The deflection angle, in radians.
    """
    v_inf = np.sqrt(2.0 * E / MU)
    if x0 is None:
        x0 = -max(200.0, 100.0 * abs(b))
    r0 = np.hypot(x0, b)
    s0 = [x0, b, v_inf, 0.0]
    t_max = 10.0 * r0 / v_inf

    def returned(_t, s, *_a):  # fires when the particle climbs back out to r0
        return np.hypot(s[0], s[1]) - r0

    returned.terminal = True
    returned.direction = 1.0  # only on the outgoing crossing

    sol = solve_ivp(
        cartesian_rhs,
        (0.0, t_max),
        s0,
        args=(k,),
        method="DOP853",
        rtol=1e-11,
        atol=1e-13,
        events=returned,
        dense_output=True,
    )
    # final (outgoing) state: the event state if it fired, else the integration end
    if sol.y_events[0].size:
        _, _, vxf, vyf = sol.y_events[0][0]
        t_end = sol.t_events[0][0]
    else:
        _, _, vxf, vyf = sol.y[:, -1]
        t_end = sol.t[-1]
    theta = np.arctan2(vyf, vxf)  # deflection from the +x incoming direction

    tg = np.linspace(0.0, t_end, n_eval)
    xg, yg, vxg, vyg = sol.sol(tg)
    rg = np.hypot(xg, yg)
    E_series = 0.5 * (vxg**2 + vyg**2) + V(rg, k)
    return theta, tg, xg, yg, E_series


def theta_analytic(b, E, k=K):
    """Closed-form Rutherford deflection angle, tan(θ/2) = |k|/(2·E·b) (eq-rutherford-angle).

    Parameters
    ----------
    b : float or numpy.ndarray
        Impact parameter.
    E : float
        Energy.
    k : float, optional
        Coupling.

    Returns
    -------
    float or numpy.ndarray
        The deflection angle.
    """
    return 2.0 * np.arctan(abs(k) / (2.0 * E * b))

## Exercise 1 — The scattering geometry

Before any numbers, fix the picture. A particle of energy $E>0$ approaches along a
line offset from the centre by the impact parameter $b$; it is repelled, swings
around the centre once, and leaves along an outgoing asymptote tilted by the
scattering angle $\theta$ from its original direction. The closer the aim (smaller
$b$), the harder the kick and the larger $\theta$. The deflection integral
{eq}`eq-theta-integral` turns this geometry into the function $\theta(b)$ we spend
the rest of the notebook exploring.

1. Integrate one scattering trajectory with `deflection` and confirm it is
   an *unbound* state: it comes in from large $r$ and returns to large $r$
   (it does not stay trapped), with $E>0$.
2. Study the geometry (incoming and outgoing asymptotes, impact parameter
   $b$, scattering angle $\theta$) drawn in {numref}`fig-scatter-geometry`.

In [ ]:
# (solution hidden on the public site)


### Validation 1 — the trajectory is an unbound scattering state

A scattering state must do two things a bound orbit never does: have $E>0$, and
*leave*: return to a large radius rather than stay trapped. We confirm the
integrated trajectory reaches back out beyond a threshold radius.

In [ ]:
th1, t1, x1, y1, E1 = deflection(1.0, 1.0)
r_final = np.hypot(x1[-1], y1[-1])
r_threshold = 50.0
print(f"final radius r = {r_final:.2f}  (threshold {r_threshold}),  E = {E1[0]:.4f}")
validate.check(
    r_final > r_threshold and E1[0] > 0,
    "the trajectory is an unbound scattering state (E>0, returns to large r)",
    f"r_final = {r_final:.1f}, E = {E1[0]:.4f}",
)

## Exercise 2 — A single scattering trajectory (worked)

Here is the scattering event in full: one particle, one impact parameter, the
hyperbolic path past the centre. The two asymptotes (the straight lines the
particle follows long before and long after the encounter) meet at the scattering
angle $\theta$. Along the way, the energy $E=\tfrac12\mu v^2+k/r$ must stay exactly
constant: the force does no net work over the round trip, it only redirects the
momentum. We integrate, draw the path with its asymptotes and $\theta$ marked, and
verify energy conservation along the orbit ({numref}`fig-scatter-trajectory`).

This is a worked exercise; you build the beam animation in Exercise 5.

In [ ]:
# (solution hidden on the public site)


### Validation 2 — energy is conserved along the trajectory

Scattering conserves energy exactly: the Coulomb force is conservative, so
$E=\tfrac12\mu v^2+k/r$ may drift only at the integrator's tolerance over the whole
encounter.

In [ ]:
validate.conserved(
    E2series, "energy conserved along the scattering trajectory", rel_drift=1e-6
)

## Exercise 3 — Deflection angle vs. impact parameter

Now the central map $b\mapsto\theta$. We fire a sweep of impact parameters at fixed
energy, measure each outgoing angle from the trajectory, and test the closed-form
law {eq}`eq-rutherford-angle`, $\tan(\theta/2)=k/(2Eb)$. The agreement is the
numerical proof that the Coulomb orbit really is the hyperbola the analysis claims.

This is also where the **finite-distance systematic** shows its face. Because we
launch from a large but *finite* $x_0$, every particle already feels the $1/r$ tail
at the start, so the measured deflection comes out a touch low, and the effect
grows with $b$ (those wide trajectories graze the tail for longer). We therefore
validate at $2\%$, and the residual creeps up with $b$ in the data:
the discrepancy is physics-of-the-method, not a coding bug. Contrast this with
Exercise 4, whose check never integrates a trajectory and so is exact.

1. At $E=1$, sweep $b$ (`numpy.linspace`) and measure $\theta(b)$ from
   `deflection`.
2. Overlay the numeric points on the analytic curve $\theta=2\arctan(k/2Eb)$
   ({numref}`fig-scatter-deflection`) and check they agree to $2\%$.

In [ ]:
# (solution hidden on the public site)


### Validation 3 — the deflection follows $\tan(\theta/2)=k/(2Eb)$

The measured angles must match the closed-form law {eq}`eq-rutherford-angle`. The
$2\%$ tolerance is set deliberately to cover the finite-launch-distance systematic
explained above: the *method's* error, not the law's.

In [ ]:
validate.close(
    theta_num,
    theta_law,
    "deflection angle follows tan(θ/2)=k/(2Eb)",
    rtol=2e-2,
)

## Exercise 4 — The differential cross-section → Rutherford (worked, centrepiece)

This is the heart of the notebook. The deflection law $b(\theta)$ is all we need:
feeding it into {eq}`eq-dsigma`, $d\sigma/d\Omega=|(b/\sin\theta)(db/d\theta)|$,
turns the single-particle geometry into the **beam** observable, and the algebra
collapses to the Rutherford formula {eq}`eq-rutherford`,
$d\sigma/d\Omega=(k/4E)^2/\sin^4(\theta/2)$.

Crucially, this exercise **never integrates a trajectory**. We use the analytic
$b(\theta)=(k/2E)\cot(\theta/2)$ and its closed-form derivative
$db/d\theta=-(k/4E)\csc^2(\theta/2)$, assemble $d\sigma/d\Omega$ from
{eq}`eq-dsigma`, and compare to the closed form. So there is no finite-distance
tail and no integrator drift: the check passes to **machine precision**, a sharp
contrast with Exercise 3's $2\%$. *That* contrast is the lesson: numerical error
enters only where one actually integrates dynamics.

1. From $b(\theta)$ and its closed-form derivative, form $d\sigma/d\Omega$
   via {eq}`eq-dsigma` and compare to the Rutherford formula
   {eq}`eq-rutherford`.
2. Plot $d\sigma/d\Omega$ vs. $\theta$ on a log axis (`semilogy`), showing
   the small-angle divergence ({numref}`fig-scatter-dsigma`).

In [ ]:
# (solution hidden on the public site)


### Validation 4 — the cross-section is the Rutherford $1/\sin^4(\theta/2)$ law

The geometric cross-section {eq}`eq-dsigma` built from $b(\theta)$ must equal the
closed-form Rutherford result {eq}`eq-rutherford` essentially exactly: this is a
pure analytic identity, with no trajectory integrated, hence the $10^{-4}$
tolerance (achieved to machine precision, $\sim10^{-15}$). Contrast Exercise 3's $2\%$.

In [ ]:
validate.close(
    dsigma_num,
    rutherford,
    "differential cross-section is the Rutherford 1/sin⁴(θ/2) law",
    rtol=1e-4,
)

## Exercise 5 — The beam picture (worked animation)

A cross-section is a statement about a *beam*, so let us fire one. Many particles
start in a vertical line (a parallel beam) at a range of impact parameters and a
common energy, and stream toward the centre. Watching them, the geometry of
scattering becomes obvious: the inner particles (small $b$) get the hardest kick
and fan out to large angles, while the outer ones sail past nearly undeflected.
That monotonic ordering, small $b\to$ large $\theta$, is exactly what concentrates
scattered particles in the forward direction and drives the Rutherford pile-up.

This is the worked animation; you build the attractive-vs-repulsive one in
Exercise 6.

In [ ]:
# (solution hidden on the public site)


### Validation 5 — energy conserved across the beam, and $b\downarrow\Rightarrow\theta\uparrow$

Two physical facts about the *animated beam data* (not the animation object): every
particle conserves its energy, and the deflection is monotonic: sorting by impact
parameter, the scattering angle decreases as $b$ grows. The monotonicity is the
mechanism behind the forward-peaked cross-section.

In [ ]:
validate.close(
    E_beam_final,
    E_beam_init,
    "every beam particle conserves energy",
    rtol=1e-4,
)
order = np.argsort(b_beam)
theta_sorted_by_b = theta_beam[order]
validate.check(
    np.all(np.diff(theta_sorted_by_b) <= 1e-6),
    "smaller impact parameter gives larger deflection (monotonic θ(b))",
    f"θ decreases from {np.degrees(theta_sorted_by_b[0]):.1f}° "
    f"to {np.degrees(theta_sorted_by_b[-1]):.1f}° as b grows",
)

## Exercise 6 — Attractive vs. repulsive (student-implemented animation)

A remarkable fact hides in the Rutherford formula: it depends on $k$ only through
$k^2$, so an **attractive** Coulomb force ($k<0$, e.g. an electron on a nucleus)
produces the *same* cross-section as the repulsive one. The deflection magnitude
obeys the same $|\tan(\theta/2)|=|k|/(2Eb)$, only the *sense* of the bend differs:
the repelled particle veers away from the centre, the attracted one swings *toward*
it and whips around the far side. The cross-section, which counts particles per
solid angle without regard to sign, cannot tell the two cases apart.

This is the **student-implemented** animation. The dynamics are handed to you (just
flip the sign of $k$); you build the side-by-side player and confirm the magnitudes
match.

1. Integrate an attractive trajectory ($k=-1$) and a repulsive one ($k=+1$)
   at the *same* $b$ and $E$ with `scipy.integrate.solve_ivp` (`DOP853`) on
   `cartesian_rhs`.
2. **Build the animation** showing both trajectories side by side, the
   attractive one bending toward the centre and the repulsive one away (many
   valid layouts exist), then `plt.close(fig)` and end with
   `ecp.animate.show(anim)`.
3. Confirm both give the same $|\theta|$
   ({numref}`fig-scatter-attract-anim`).

> A ✗ on the final check points at the **deflection measurements or starting
> distance** (start far enough that the finite-$x_0$ tail is small for *both*
> signs), not at the drawing: any correct animation of the same data is fine.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6 — attraction and repulsion give the same $|\theta|$

The signature of the $k^2$ dependence: at equal $b$ and $E$, the attractive and
repulsive deflection *magnitudes* agree. The $2\%$ tolerance again covers the
finite-launch-distance systematic (common to both signs).

In [ ]:
validate.close(
    abs(th_att),
    abs(th_rep),
    "attractive and repulsive Coulomb give the same deflection magnitude",
    rtol=2e-2,
)

## Exercise 7 — Total cross-section and the small-angle catastrophe

The total cross-section integrates the differential one over all directions,
$\sigma=\int (d\sigma/d\Omega)\,d\Omega=\int_0^\pi (d\sigma/d\Omega)\,2\pi\sin\theta\,d\theta$.
For Rutherford {eq}`eq-rutherford` the integrand behaves like $1/\theta^3$ near
$\theta=0$, so the integral **diverges** at the forward end: the partial total from
a small cutoff $\theta_{\min}$ to $\pi$ grows without bound as $\theta_{\min}\to0$.
Physically this is the infinite range of the bare Coulomb force: *every* particle,
at *every* impact parameter, is deflected a little, so there is no finite "size" to
the target. A real charge is screened beyond some radius $a$ (the forward link:
the Yukawa potential $V=k\,e^{-r/a}/r$, and the quantum Born approximation of
Volume VI), which cuts off small angles and restores a finite $\sigma$.

As a closing sanity check we also confirm the **small-angle limit** of the
deflection law: for large $b$, $\tan(\theta/2)\approx\theta/2$, so $\theta\approx
k/(Eb)$.

1. Compute the partial total cross-section $\sigma(\theta_{\min})$
   (`numpy.trapezoid` over the angular grid) for a decreasing sequence of
   cutoffs and show it grows without bound ({numref}`fig-scatter-total`).
2. Verify $\theta\approx k/(Eb)$ at large $b$ against a measured trajectory.

In [ ]:
# (solution hidden on the public site)


### Validation 7 — the total cross-section diverges, and $\theta\approx k/(Eb)$

Two checks. First, the partial total cross-section strictly *increases* as the
forward cutoff $\theta_{\min}$ shrinks: the Coulomb total has no finite limit.
Second, the large-$b$ small-angle limit $\theta\approx k/(Eb)$ holds against a
measured trajectory (the $4\%$ tolerance covers the finite-distance systematic,
which is largest exactly here, at the widest $b$).

In [ ]:
validate.check(
    np.all(np.diff(sigmas) > 0),  # σ rises monotonically as θ_min falls
    "the Coulomb total cross-section diverges (infinite range)",
    f"σ grows {sigmas[-1] / sigmas[0]:.0f}× as θ_min: {theta_mins[0]}→{theta_mins[-1]}",
)
validate.close(
    theta_small_num,
    theta_small_pred,
    "small-angle deflection follows θ ≈ k/(Eb) for large b",
    rtol=4e-2,
)

## Notebook summary

- The scattering geometry and single trajectories integrated through a repulsive potential;
  the deflection angle as a function of impact parameter.
- The differential cross-section assembled from $d\sigma/d\Omega=(b/\sin\theta)|db/d\theta|$,
  reproducing the **Rutherford** $1/\sin^4(\theta/2)$ law; attractive versus repulsive
  scattering; and the small-angle divergence of the total cross-section for the Coulomb tail.

## Outlook

- **Screened Coulomb (Yukawa).** $V(r)=k\,e^{-r/a}/r$ cuts off the force beyond the
  screening length $a$, giving a *finite* total cross-section; integrate $\theta(b)$
  numerically and watch the small-angle divergence tamed.
- **Rainbow and glory angles.** An attractive potential with a softer core can make
  $d\theta/db$ vanish at some $b$ ($db/d\theta\to\infty$ in {eq}`eq-dsigma`), a
  *rainbow* singularity in $d\sigma/d\Omega$, the classical analogue of the optical
  rainbow; $\theta\to0$ or $\pi$ at nonzero $b$ gives a *glory*.
- **Hard-sphere scattering.** The analytic warm-up: a rigid sphere of radius $R$
  gives the isotropic $d\sigma/d\Omega=R^2/4$ and finite total $\sigma=\pi R^2$: the
  geometric cross-section, with no forward divergence.
- **The quantum cross-section.** The Born approximation for the Coulomb potential
  reproduces the Rutherford formula *exactly*: a striking classical/quantum
  agreement we return to in Volume VI.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()